1. 데이터 로드 및 전처리

In [ ]:
!pip install pandas

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity


# JSON 파일로부터 사용자 스타일 벡터 추출
def load_user_styles(json_dir):
    user_data = []

    # JSON 디렉터리 내 모든 파일 순회
    for file_name in os.listdir(json_dir):
        if file_name.endswith(".json"):
            file_path = os.path.join(json_dir, file_name)
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

                # 사용자 ID와 스타일 벡터 추출
                user_info = data['user']
                user_id = user_info['R_id']
                style_vector = [
                    user_info['r_style1'], user_info['r_style2'],
                    user_info['r_style3'], user_info['r_style4'],
                    user_info['r_style5']
                ]
                user_data.append({"user_id": user_id, "style_vector": style_vector})

    # DataFrame으로 변환
    df = pd.DataFrame(user_data)
    return df

# 데이터 경로 설정
train_json_dir = r"data/training_label"  # 실제 경로로 변경

# 사용자 데이터 로드
df = load_user_styles(train_json_dir)
print(df.head())  # 데이터 확인


In [ ]:
# 사용자 스타일 벡터를 NumPy 배열로 변환
user_matrix = np.vstack(df['style_vector'].values)

# 코사인 유사도 계산
user_similarity = cosine_similarity(user_matrix)

# 유사도 매트릭스를 DataFrame으로 변환
user_similarity_df = pd.DataFrame(
    user_similarity, index=df['user_id'], columns=df['user_id']
)

print(user_similarity_df.head())  # 유사도 확인


In [ ]:
# 추천 스타일 예측 함수
def recommend_styles(user_id, user_similarity_df, df, n_recommendations=3):
    # 유사한 사용자 추출 (자기 자신 제외)
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:]

    # 유사한 사용자의 스타일 벡터 가져오기
    recommended_styles = []
    for uid in similar_users.index:
        styles = df.loc[df['user_id'] == uid, 'style_vector'].values[0]
        recommended_styles.extend(styles)

    # 최다 등장한 상위 N개 스타일 반환
    unique, counts = np.unique(recommended_styles, return_counts=True)
    top_recommendations = unique[np.argsort(counts)[-n_recommendations:]]

    return top_recommendations

# 예제: 사용자 64747에게 추천
recommended = recommend_styles(64747, user_similarity_df, df)
print(f"추천 스타일: {recommended}")


In [ ]:
# Validation JSON 경로 설정
valid_json_dir = r"data/validation_label"  # 실제 경로로 변경

# Validation 데이터에서 해당 사용자의 실제 스타일 추출
def get_actual_styles(user_id, json_dir):
    for file_name in os.listdir(json_dir):
        if file_name.endswith(".json"):
            file_path = os.path.join(json_dir, file_name)
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                if data['user']['R_id'] == user_id:
                    return [
                        data['user']['r_style1'], data['user']['r_style2'],
                        data['user']['r_style3'], data['user']['r_style4'],
                        data['user']['r_style5']
                    ]
    return []

# 정확도 계산 함수
def calculate_accuracy(user_id, recommended, json_dir):
    actual_styles = get_actual_styles(user_id, json_dir)
    correct = sum([1 for style in recommended if style in actual_styles])
    accuracy = correct / len(recommended) if recommended else 0
    return accuracy

# 사용자 64747의 정확도 계산
accuracy = calculate_accuracy(64747, recommended, valid_json_dir)
print(f"추천 정확도: {accuracy:.2%}")
